# Figure 3 — the primary outcome

**What happened.** The primary quantity is one number, the common odds ratio of a
better 90-day mRS under bridging `[§8]`. It rests on a constant-shift assumption that `[§8]`
prescribes no test of, and the mitigation `[§8]` prescribes instead is the six cumulative risk
differences. Where those six do not share a sign, an odds ratio near the null is **not** a null
result, and `[§16a]` requires the reader to be told so. A figure says it; a single-row table
hides it.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The weighted distribution, and the six differences

`report.pmf` turns the cumulative table the estimator produced into per-category shares — the same
function `F01` and `T08` use, so panel A cannot disagree with them.

In [ ]:
shares = {a: report.pmf(bundle.primary.cumulative, a) for a in ARMS}

rd = pd.DataFrame(
    [(k, bundle.primary.cumulative[k][0], bundle.primary.cumulative[k][1], bundle.primary.rd[k],
      bundle.boot.intervals[f"rd_{k}"].lo, bundle.boot.intervals[f"rd_{k}"].hi)
     for k in C.MRS_THRESHOLDS],
    columns=["k", f"P(mRS<=k) {ARM[0]}", f"P(mRS<=k) {ARM[1]}", "RD_k", "ci lo", "ci hi"])
rd

In [ ]:
beta = bundle.boot.intervals["beta"]
odds_ratio = bundle.primary.odds_ratio
summary = (f"common odds ratio {odds_ratio:.2f}  "
           f"(95% CI {np.exp(beta.lo):.2f} to {np.exp(beta.hi):.2f};  p = {beta.p:.3f})")
print(summary)
print(f"the six RD_k share a sign: {len({np.sign(v) for v in bundle.primary.rd.values()}) == 1}")

## 3. The figure

Panel A is the distribution the odds ratio summarises. Panel B is what the summary costs: six
threshold-specific differences, on the absolute scale, with intervals and **no p-values** — testing six
cuts of one contrast invites picking the most favourable `[§8]`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6.6, 5.0), gridspec_kw={"height_ratios": [1, 1.7]})

M.stacked_mrs(axes[0], [(ARM[a], shares[a]) for a in ARMS])
axes[0].legend(ncol=7, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.45))

rows = [(f"mRS ≤ {k}", bundle.primary.rd[k], bundle.boot.intervals[f"rd_{k}"].lo,
         bundle.boot.intervals[f"rd_{k}"].hi, M.Mark.ESTIMATE) for k in C.MRS_THRESHOLDS]
M.forest(axes[1], rows, f"risk difference  ({ARM[1]} − {ARM[0]})", 0.0)

# Only the letters go on the canvas. What each panel shows, and the odds ratio itself, are the
# legend's job — and the legend is not printed into the PNG.
fig.tight_layout()
fig.subplots_adjust(hspace=0.9)      # room for the mRS key beneath A, clear of B
M.panel_letters(fig, {"A": axes[0], "B": axes[1]}, dy=0.015)

## 4. Write it

`M.primary_clauses` is the whole of what `[§16]` requires wherever `exp(β)` appears, computed from
the run: the constant-shift statement, the direction statement **if** the six differences disagree,
the residual imbalance named with its magnitude, and how the intervals were made.

In [ ]:
caption = (
    "Figure 3. Primary 90-day outcome in the [§7] overlap population. (A) Overlap-weighted "
    "distribution of the modified Rankin Scale by arm. (B) Cumulative risk differences RD_0 to RD_5 "
    f"with 95% percentile intervals and no p-values [§8]; {summary}.")

source = pd.DataFrame(
    [("A", ARM[a], f"P(mRS = {j})", shares[a][j]) for a in ARMS for j in C.MRS_LEVELS]
    + [("B", f"mRS <= {k}", quantity, value)
       for k in C.MRS_THRESHOLDS
       for quantity, value in (("RD_k", bundle.primary.rd[k]),
                               ("ci lo", bundle.boot.intervals[f"rd_{k}"].lo),
                               ("ci hi", bundle.boot.intervals[f"rd_{k}"].hi))]
    + [("B", "exp(beta)", quantity, value)
       for quantity, value in (("estimate", odds_ratio), ("ci lo", np.exp(beta.lo)),
                               ("ci hi", np.exp(beta.hi)), ("p", beta.p))],
    columns=["panel", "row", "quantity", "value"])

path = M.save_figure("Figure_3", fig, source, (caption, *M.primary_clauses(bundle),
                                               M.provenance_clause(bundle)))
M.save_legends()      # rebuilds out/manuscript/figure_legends.md from every figure on disk
M.show(path)

## 5. Reading it

- **Read panel B before panel A's summary.** The odds ratio is one shared shift fitted to six cuts.
  When the cuts disagree in direction, that number is a statement about the absence of a *uniform*
  shift, not about the absence of a difference `[§16a]`.
- **The intervals include the cost of estimating the weights.** Every replicate refits the propensity
  model and the ordinal model `[§10]`; a model-based standard error would pretend the weights were
  known.
- **No p-value appears on the absolute scale, by design** `[§8]`. The primary outcome carries exactly
  one test, on `β`.
- **A covariate inside the model is still imbalanced**, and the caption names it. The E-value bounds
  *unmeasured* confounding and is reported in the supplement `[§16d]`; it does not address this.